In [ ]:
!pip install -q transformers sentencepiece sacrebleu pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 5.4 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd
import sacrebleu

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("NLLB model loaded successfully!")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

NLLB model loaded successfully!


In [3]:
def translate_sanskrit(text):
    tokenizer.src_lang = "san_Deva"

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),
        max_length=100
    )

    translation = tokenizer.batch_decode(
        translated_tokens,
        skip_special_tokens=True
    )[0]

    return translation

In [4]:
sanskrit_sentences = [
    "रामः विद्यालयं गच्छति।",
    "सीता पुस्तकं पठति।",
    "बालकाः उद्याने क्रीडन्ति।"
]

for sentence in sanskrit_sentences:
    english = translate_sanskrit(sentence)

    print("Sanskrit :", sentence)
    print("English  :", english)
    print()

Sanskrit : रामः विद्यालयं गच्छति।
English  : Ram goes to school.

Sanskrit : सीता पुस्तकं पठति।
English  : Sita is reading books.

Sanskrit : बालकाः उद्याने क्रीडन्ति।
English  : Children play in the garden.



In [5]:
data = {
    "Sanskrit": [
        "रामः विद्यालयं गच्छति।",
        "सीता पुस्तकं पठति।",
        "बालकाः उद्याने क्रीडन्ति।",
        "सूर्यः पूर्वदिशायाम् उदेति।",
        "कृषकः क्षेत्रे कार्यं करोति।"
    ],
    "English": [
        "Rama goes to school.",
        "Sita reads a book.",
        "The children play in the garden.",
        "The sun rises in the east.",
        "The farmer works in the field."
    ]
}

df = pd.DataFrame(data)

df.to_csv("sanskrit_test.csv", index=False)

print("Test CSV created successfully!")
print(df)

Test CSV created successfully!
                       Sanskrit                           English
0        रामः विद्यालयं गच्छति।              Rama goes to school.
1            सीता पुस्तकं पठति।                Sita reads a book.
2     बालकाः उद्याने क्रीडन्ति।  The children play in the garden.
3   सूर्यः पूर्वदिशायाम् उदेति।        The sun rises in the east.
4  कृषकः क्षेत्रे कार्यं करोति।    The farmer works in the field.


In [6]:
df = pd.read_csv("sanskrit_test.csv")

print("Test dataset:")
print(df)

Test dataset:
                       Sanskrit                           English
0        रामः विद्यालयं गच्छति।              Rama goes to school.
1            सीता पुस्तकं पठति।                Sita reads a book.
2     बालकाः उद्याने क्रीडन्ति।  The children play in the garden.
3   सूर्यः पूर्वदिशायाम् उदेति।        The sun rises in the east.
4  कृषकः क्षेत्रे कार्यं करोति।    The farmer works in the field.


In [7]:
model_outputs = []

for sentence in df["Sanskrit"]:
    translation = translate_sanskrit(sentence)
    model_outputs.append(translation)

df["Model Output English"] = model_outputs

print("Model translations:")
print(df)

Model translations:
                       Sanskrit                           English  \
0        रामः विद्यालयं गच्छति।              Rama goes to school.   
1            सीता पुस्तकं पठति।                Sita reads a book.   
2     बालकाः उद्याने क्रीडन्ति।  The children play in the garden.   
3   सूर्यः पूर्वदिशायाम् उदेति।        The sun rises in the east.   
4  कृषकः क्षेत्रे कार्यं करोति।    The farmer works in the field.   

                   Model Output English  
0                   Ram goes to school.  
1                Sita is reading books.  
2          Children play in the garden.  
3            The sun rises in the east.  
4  The farmer is working in the fields.  


In [8]:
df.to_csv("sanskrit_translation_results.csv", index=False)

print("Translation results saved successfully!")

Translation results saved successfully!


In [9]:
bleu_scores = []
chrf_scores = []

for i in range(len(df)):

    ground_truth = df.loc[i, "English"]
    model_output = df.loc[i, "Model Output English"]

    bleu = sacrebleu.sentence_bleu(
        model_output,
        [ground_truth]
    )

    chrf = sacrebleu.sentence_chrf(
        model_output,
        [ground_truth]
    )

    bleu_scores.append(bleu.score)
    chrf_scores.append(chrf.score)

df["BLEU"] = bleu_scores
df["CHRF"] = chrf_scores

print("BLEU and CHRF scores calculated!")
print(df)

BLEU and CHRF scores calculated!
                       Sanskrit                           English  \
0        रामः विद्यालयं गच्छति।              Rama goes to school.   
1            सीता पुस्तकं पठति।                Sita reads a book.   
2     बालकाः उद्याने क्रीडन्ति।  The children play in the garden.   
3   सूर्यः पूर्वदिशायाम् उदेति।        The sun rises in the east.   
4  कृषकः क्षेत्रे कार्यं करोति।    The farmer works in the field.   

                   Model Output English        BLEU        CHRF  
0                   Ram goes to school.   66.874030   79.250912  
1                Sita is reading books.   12.703319   36.057625  
2          Children play in the garden.   64.318702   85.701992  
3            The sun rises in the east.  100.000000  100.000000  
4  The farmer is working in the fields.   16.515822   66.787794  


In [10]:
df.to_csv(
    "final_sanskrit_translation_results.csv",
    index=False
)

print("Final CSV created successfully!")
print()
print(df)

Final CSV created successfully!

                       Sanskrit                           English  \
0        रामः विद्यालयं गच्छति।              Rama goes to school.   
1            सीता पुस्तकं पठति।                Sita reads a book.   
2     बालकाः उद्याने क्रीडन्ति।  The children play in the garden.   
3   सूर्यः पूर्वदिशायाम् उदेति।        The sun rises in the east.   
4  कृषकः क्षेत्रे कार्यं करोति।    The farmer works in the field.   

                   Model Output English        BLEU        CHRF  
0                   Ram goes to school.   66.874030   79.250912  
1                Sita is reading books.   12.703319   36.057625  
2          Children play in the garden.   64.318702   85.701992  
3            The sun rises in the east.  100.000000  100.000000  
4  The farmer is working in the fields.   16.515822   66.787794  


In [11]:
from google.colab import files

files.download("final_sanskrit_translation_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# Overall BLEU and CHRF score

ground_truth = df["English"].tolist()
model_output = df["Model Output English"].tolist()

bleu = sacrebleu.corpus_bleu(
    model_output,
    [ground_truth]
)

chrf = sacrebleu.corpus_chrf(
    model_output,
    [ground_truth]
)

print("Overall BLEU Score :", bleu.score)
print("Overall CHRF Score :", chrf.score)

Overall BLEU Score : 54.64629269807536
Overall CHRF Score : 75.86486073754507
